In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
!pip list

In [51]:
from embeddings import load_embedding_vector_store 

In [3]:
import embeddings as emb
import inference as inf
import boto3
from pathlib import Path

In [17]:
import sys
import os
from dotenv import load_dotenv

In [18]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [19]:
load_dotenv(override=True)

True

In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
)

In [ ]:
bedrock_client = boto3.client(service_name="bedrock-runtime", 
                              region_name="us-east-1")
system_prompt = [{"text": "You are a helpful assistant who answers user questions ans also uses previous user interactions with you to answer the questions."}]
messages = [{
                "role": "user",
                "content": [{"text": "I am a girl who loves the color blue."}]
            }
           ]

response = bedrock_client.converse(
            modelId="openai.gpt-oss-120b-1:0",
            messages=messages,
            system=system_prompt)

response

In [ ]:
messages.append(response['output']['message'])
messages

In [13]:
messages.append({
                "role": "user",
                "content": [{"text": "Why is blue my favorite color?"}]
            })

response = bedrock_client.converse(
            modelId="openai.gpt-oss-120b-1:0",
            messages=messages,
            system=system_prompt)

messages.append(response['output']['message'])
messages

[{'role': 'user',
  'content': [{'text': 'I am a girl who loves the color blue.'}]},
 {'role': 'assistant',
  'content': [{'reasoningContent': {'reasoningText': {'text': 'The user says: "I am a girl who loves the color blue." There\'s no question, just a statement. Perhaps they want a response, maybe an engaging reply. Should respond in friendly manner, maybe ask about why they love blue, or suggest things. Should keep safe. Use prior interactions? There\'s none besides this. So respond.'}}},
   {'text': 'That’s wonderful! 💙 Blue is such a calming and versatile color—whether it’s the deep navy of a night sky, the bright turquoise of tropical waters, or the soft pastel of a spring morning, there’s a shade for every mood.\n\nDo you have a favorite shade of blue, or maybe something special that’s blue (like a piece of clothing, a hobby, or a place) that makes you feel extra happy? I’d love to hear more about what draws you to this lovely color!'}]},
 {'role': 'user', 'content': [{'text': 

In [ ]:
messages = [
    (
        "system",
        "You are a helpful assistant that knows about zoning codes.",
    ),
    ("human", "What is a zoning code?"),
]
ai_msg = llm.invoke(messages)
ai_msg

In [14]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_by_title")



Loading locally available vector store.


In [59]:
from langchain_core.messages import HumanMessage, AIMessage
from prompts import SYSTEM_PROMPT

# 1. Create prompt with placeholders for context, history, and query
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])


# 2. Create model and chain
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
)
chain = qa_prompt | llm

# 3. Use it with your retriever and chat history
chat_history = []

# Retrieve docs
query = "zoning info **2817 Pleasant Ave, Minneapolis, MN 55408** multi-family home, I'm a professional developer"
retrieved_docs = e5_large_unstruct_by_title.similarity_search(query)
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Invoke with all three: context, history, and query
response = chain.invoke({
    "context": context,
    "chat_history": chat_history,
    "question": query
})

In [60]:
response


AIMessage(content='Based on the retrieved zoning information for the address 2817 Pleasant Ave, Minneapolis, MN 55408, which pertains to multi-family homes, since there is not a specific mention of multi-family homes in the provided context, it is unclear how the zoning regulations apply in this case. Unfortunately, without additional information specific to multi-family homes at this address, I am unable to provide detailed zoning information.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 1899, 'total_tokens': 1980, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CbBqDih2kShYkCmf0whNc1WmJu4rm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--1933a531-

In [43]:
chat_history = [
    HumanMessage(content = "zoning info **2817 Pleasant Ave, Minneapolis, MN 55408** multi-family home, I'm a professional developer"),
    AIMessage(content = response.content)
]

# Retrieve docs
query = "zoning info **2817 Pleasant Ave, Minneapolis, MN 55408** multi-family home, I'm a professional developer"
retrieved_docs = e5_large_unstruct_by_title.similarity_search(query)
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Invoke with all three: context, history, and query
response = chain.invoke({
    "context": context,
    "chat_history": chat_history,
    "input": query
})

In [46]:
response.content

"I don't have specific information on the zoning district returned by the previous question to provide a summary of the zoning ordinance for that district in Minneapolis, MN."

In [56]:

from prompts import SYSTEM_PROMPT

llm = ChatOpenAI()
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reformulate the question based on chat history"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])

history_aware_retriever = create_history_aware_retriever(
    llm, e5_large_unstruct_by_title.as_retriever(), contextualize_prompt
)

# 3. Create QA chain
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])
qa_chain = create_stuff_documents_chain(llm, qa_prompt)

# 4. Combine into RAG chain
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

# 5. Use it
chat_history = []
response = rag_chain.invoke({"question": "zoning info **2817 Pleasant Ave, Minneapolis, MN 55408** multi-family home, I'm a professional developer", "chat_history": chat_history})

ValueError: Expected `input` to be a prompt variable, but got ['chat_history', 'question']

In [ ]:
response

In [28]:
chat_history.append( {"type": "human", "content": "How is a single family home located at **2712 Colfax Ave South, Minneapolis, MN 55408** zoned?"})
chat_history.append( {"type": "ai", "content": response["answer"]})
chat_history

[{'type': 'human',
  'content': 'How is a single family home located at **2712 Colfax Ave South, Minneapolis, MN 55408** zoned?'},
 {'type': 'ai',
  'content': 'Based on the zoning code information retrieved, the property at 2817 Pleasant Ave, Minneapolis falls under the category of a multi-family home with four units or more. As a professional developer, you may need to adhere to specific regulations and guidelines outlined for multiple-family dwellings in Minneapolis.'}]

In [31]:
response = rag_chain.invoke({"input": "summarize zoning ordinance for [zoning district returned by question 1], Minneapolis, MN", "chat_history": chat_history})


In [32]:
chat_history.append( {"type": "human", "content": "summarize zoning ordinance for [zoning district returned by question 1], Minneapolis, MN"})
chat_history.append( {"type": "ai", "content": response["answer"]})
chat_history

[{'type': 'human',
  'content': 'How is a single family home located at **2712 Colfax Ave South, Minneapolis, MN 55408** zoned?'},
 {'type': 'ai',
  'content': 'Based on the zoning code information retrieved, the property at 2817 Pleasant Ave, Minneapolis falls under the category of a multi-family home with four units or more. As a professional developer, you may need to adhere to specific regulations and guidelines outlined for multiple-family dwellings in Minneapolis.'},
 {'type': 'human',
  'content': "zoning info **2817 Pleasant Ave, Minneapolis, MN 55408** multi-family home, I'm a professional developer"},
 {'type': 'ai',
  'content': 'Based on the zoning information provided, the property at 2817 Pleasant Ave, Minneapolis is classified as a multi-family home. As a professional developer, you should be aware of the regulations and requirements that apply to multi-family dwellings in that location.'},
 {'type': 'human',
  'content': 'summarize zoning ordinance for [zoning district 

## Load vector stores from S3

In [62]:
# Load recursive split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_recursive = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="recursive")

Loading locally available vector store.


In [4]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_basic")

Loading locally available vector store.


In [5]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_by_title")

Loading locally available vector store.


In [6]:
# Load recursive split embeddings for model "intfloat/e5-small-v2"
e5_small_recursive = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="recursive")

Loading locally available vector store.


In [7]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_basic")

Loading locally available vector store.


In [8]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_by_title")

Loading locally available vector store.


## Get user queries from AWS S3 

In [61]:
# Get user queries from S3
base_dir = Path.cwd().parent 
local_file_dir = base_dir / "temp" 
queries = emb.download_from_s3(s3_path="zoning-code-questions.csv", output_path=local_file_dir)

Saved file from s3://local-gov-ai-llm-benchmarking/zoning-code-questions.csv at /home/jovyan/llm-benchmarking/temp.


## Run Inference using AWS Converse API

In [63]:
# Define dictionary for embedding models 
embeddings = {
    "e5_large_recursive" : e5_large_recursive, 
    "e5_large_unstruct_basic" : e5_large_unstruct_basic,
    "e5_large_unstruct_by_title" : e5_large_unstruct_by_title,  
    "e5_small_recursive" : e5_small_recursive,  
    "e5_small_unstruct_basic" : e5_small_unstruct_basic,
    "e5_small_unstruct_by_title" : e5_small_unstruct_by_title
}

# Define list for question types
question_types = ["comp1-aud1-data1", "comp2-aud1-data1", "comp3-aud1-data1", "comp1-aud2-data1", "comp2-aud2-data1", "comp3-aud2-data1"]

In [64]:
# Create output folder
local_path = base_dir / "output"
local_path.mkdir(exist_ok=True)

### Meta Llama 3.2 1B Instruct

In [12]:
# Define model inference profile name
llama_inf_profile_name = "US Meta Llama 3.2 1B Instruct"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "meta-llama" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.run_conversation(llama_inf_profile_name, q_type, emb_name, emb_vs, path, n_iter=3)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/meta-llama/{emb_name}") 


Running conversation for 'comp1-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp1-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Uplo

### Mistral AI Pixtral Large

In [13]:
# Define model inference profile name
mistral_inf_profile_name = "US Mistral Pixtral Large 25.02"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "mistral-ai-pixtral" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.run_conversation(mistral_inf_profile_name, q_type, emb_name, emb_vs, path)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/mistral-ai-pixtral/{emb_name}") 
        

Running conversation for 'comp1-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp1-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Uplo

### OpenAI GPT OSS 120B

In [14]:
# Define model id
openai_gpt_model_id = "openai.gpt-oss-120b-1:0"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "openai-gpt-oss" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.run_conversation(openai_gpt_model_id, q_type, emb_name, emb_vs, path)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/mistral-ai-pixtral/{emb_name}") 
        

Running conversation for 'comp1-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp1-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp2-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Running conversation for 'comp3-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Conversation iteration 2
Conversation iteration 3
Uplo

In [70]:
# Define model id
openai_gpt_model_id = "gpt-5-mini"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "test" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.rag_with_openai(openai_gpt_model_id, q_type, emb_name, emb_vs, path, n_iter=1)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/test/{emb_name}") 
        

Running conversation for 'comp1-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Running conversation for 'comp2-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Running conversation for 'comp3-aud1-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Running conversation for 'comp1-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Running conversation for 'comp2-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Running conversation for 'comp3-aud2-data1' questions using e5_large_recursive embeddings.
Conversation iteration 1
Uploaded file from /home/jovyan/llm-benchmarking/output/test/e5_large_recursive/output_comp1-aud1-data1_2025-11-12.csv to s3://local-gov-ai-llm-benchmarking/output/test/e5_large_recursive
Uploaded file from /home/jovyan/llm-benchmarking/output/test/e5_large_recursive/output_comp2-aud1-data1_2025-11-12.c